# ส่วนที่ 4 · Fine-tune mT5 สรุปข่าวภาษาไทย (Abstractive)

**ผู้รับผิดชอบ:** คนที่ 4 (คู่ C) · **ส่ง:** โมเดลที่ดีที่สุด + ROUGE

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** แล้ว Runtime → **Run all**

- ข้อมูล: [ThaiSum](https://huggingface.co/datasets/pythainlp/thaisum) โหลดแบบ streaming ไม่ต้องโหลดทั้ง 350,000 ข่าว
- โมเดลเริ่มต้น: `csebuetnlp/mT5_multilingual_XLSum` เป็น mT5 ที่เคยเทรนสรุปข่าวหลายภาษารวมภาษาไทยมาแล้ว (XL-Sum, Hasan et al. 2021) เราเทรนต่อด้วยข่าวไทย
- ถ้าคนที่ 1 แจกไฟล์ train/val/test แล้ว ให้ตั้ง `TRAIN_CSV` / `VAL_CSV` / `TEST_CSV` (คอลัมน์ `body`, `summary`)

**คนที่ 5:** เปลี่ยน `MODEL_NAME = "facebook/mbart-large-50"` (ดูหมายเหตุ mBART ในช่องตั้งค่า) แล้ว Run all

In [1]:
!pip -q install transformers datasets sentencepiece accelerate rouge_score pythainlp

## 1. ตั้งค่า (แก้แค่ช่องนี้)

In [2]:
MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"   # หรือ "google/mt5-small" (เริ่มจากศูนย์ ผลแย่กว่า แต่เร็ว)
# mBART (คนที่ 5): "facebook/mbart-large-50" + ตั้ง MBART_LANG = "th_TH"
MBART_LANG = None

# ข้อมูล: None = โหลด ThaiSum จาก Hugging Face · ใส่ path = ใช้ไฟล์ของคนที่ 1
TRAIN_CSV, VAL_CSV, TEST_CSV = None, None, None
TRAIN_SIZE = 5000    # จำนวนข่าวที่ใช้เทรน (Colab T4 ~5,000 ข่าว/epoch ประมาณ 30–40 นาที)
VAL_SIZE   = 300
TEST_SIZE  = 500     # ทุกโมเดลต้องใช้ test ชุดเดียวกัน

MAX_INPUT  = 512     # ความยาวข่าว (token) ที่ป้อนเข้า
MAX_TARGET = 128     # ความยาวบทสรุปสูงสุด
EPOCHS     = 2
LR         = 1e-4    # mt5-small เริ่มจากศูนย์ใช้ 5e-4 · mBART ใช้ 3e-5
BATCH_SIZE = 4
GRAD_ACC   = 4       # batch จริง = 4 x 4 = 16
NUM_BEAMS  = 4
OUTPUT_DIR = "mt5-thaisum"
SEED = 42

## 2. โหลดข้อมูล

In [3]:
import pandas as pd
from datasets import load_dataset, Dataset

def from_thaisum(split, n, shuffle=False):
    ds = load_dataset("pythainlp/thaisum", split=split, streaming=True)
    if shuffle:
        ds = ds.shuffle(seed=SEED, buffer_size=10_000)   # สุ่มให้ได้ข่าวหลายสำนัก
    return pd.DataFrame(list(ds.take(n)))[["body", "summary"]]

def load(csv, split, n, shuffle=False):
    df = pd.read_csv(csv) if csv else from_thaisum(split, n, shuffle)
    df = df.dropna(subset=["body", "summary"])
    return df[(df.body.str.len() > 0) & (df.summary.str.len() > 0)].reset_index(drop=True)

train_df = load(TRAIN_CSV, "train", TRAIN_SIZE, shuffle=True)
val_df   = load(VAL_CSV, "validation", VAL_SIZE)
test_df  = load(TEST_CSV, "test", TEST_SIZE)
print("train", len(train_df), "| val", len(val_df), "| test", len(test_df))
print("\nตัวอย่างข่าว:", train_df.body[0][:200], "...\nบทสรุป:", train_df.summary[0])

d:\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train 5000 | val 300 | test 500

ตัวอย่างข่าว: หลังจากที่ แทค ภรัณยู ควงแฟนสาวนอกวงการ เบลล์ อดิราภ์ เองตระกูล วิวาห์ฟ้าแลบเมื่อวันอังคารที่ผ่านมา จนกลายเป็นข่าวฮือฮาที่หลายคนให้ความสนใจ และนอกจากจะสละโสดแล้ว หนุ่มแทคยังมีข่าวดีอีกเรื่องคือ ไฮโซเบ ...
บทสรุป: มดดำ เผย แทค ภรัณยู คบไฮโซเบลล์ มานานแล้ว ก่อนที่จะคบกับ ดีเจต้นหอม พร้อมบอกฝ่ายหญิงอดทนหนักมาก แม้ว่า แทคจะไปเรื่อยๆ กับคนอื่น


## 3. Tokenize

In [4]:
from transformers import AutoTokenizer
tok_kwargs = {"src_lang": MBART_LANG, "tgt_lang": MBART_LANG} if MBART_LANG else {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tok_kwargs)

def prep(batch):
    x = tokenizer(batch["body"], max_length=MAX_INPUT, truncation=True)
    x["labels"] = tokenizer(text_target=batch["summary"], max_length=MAX_TARGET, truncation=True)["input_ids"]
    return x

train_ds = Dataset.from_pandas(train_df).map(prep, batched=True, remove_columns=["body", "summary"])
val_ds   = Dataset.from_pandas(val_df).map(prep, batched=True, remove_columns=["body", "summary"])

lens = [len(tokenizer(b).input_ids) for b in train_df.body[:200]]
print(f"ความยาวข่าวเฉลี่ย {sum(lens)/len(lens):.0f} token · ถูกตัดเกิน {MAX_INPUT}: {sum(l > MAX_INPUT for l in lens)/len(lens):.0%}")

Map: 100%|██████████| 300/300 [00:00<00:00, 322.17 examples/s]


ความยาวข่าวเฉลี่ย 704 token · ถูกตัดเกิน 512: 45%


## 4. ROUGE สำหรับภาษาไทย
ไลบรารี ROUGE แยกคำด้วยช่องว่าง ภาษาไทยจึงต้องตัดคำด้วย PyThaiNLP ก่อน (ดู README: ทุกคนใช้ฟังก์ชันเดียวกันนี้)

In [5]:
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

class ThaiTokenizer:
    def tokenize(self, text):
        return [w for w in word_tokenize(text, engine="newmm") if w.strip()]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

def thai_rouge(preds, refs):
    s = [scorer.score(r, p) for p, r in zip(preds, refs)]
    return {k: round(100 * sum(x[k].fmeasure for x in s) / len(s), 2) for k in ["rouge1", "rouge2", "rougeL"]}

print(thai_rouge(["นายกรัฐมนตรีประชุมครม.วันนี้"], ["วันนี้นายกรัฐมนตรีเป็นประธานประชุมครม."]))

{'rouge1': 80.0, 'rouge2': 25.0, 'rougeL': 60.0}


## 5. ลองก่อนเทรน (zero-shot)
เอาไว้เทียบว่าการ fine-tune ช่วยแค่ไหน

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "ไม่มี (ช้ามาก ให้เปิด T4 GPU)")

def summarize(texts, model, bs=8):
    out = []
    model.eval()
    gen_kwargs = {"forced_bos_token_id": tokenizer.lang_code_to_id[MBART_LANG]} if MBART_LANG else {}
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i+bs], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=NUM_BEAMS,
                                 no_repeat_ngram_size=3, length_penalty=1.0, **gen_kwargs)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return out

from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
N0 = min(100, len(test_df))
zero = summarize(test_df.body[:N0].tolist(), model)
print("zero-shot ROUGE (100 ข่าวแรกของ test):", thai_rouge(zero, test_df.summary[:N0].tolist()))
print("\nตัวอย่าง:", zero[0])

GPU: ไม่มี (ช้ามาก ให้เปิด T4 GPU)


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 9016.25it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 6. Fine-tune
> mT5 ใช้ fp16 แล้วค่า loss มักเป็น NaN จึงเทรนแบบ fp32 + Adafactor + gradient checkpointing เพื่อให้พอกับ GPU ของ Colab
> ถ้าเจอ `CUDA out of memory` ให้ลด `BATCH_SIZE` เป็น 2 แล้วเพิ่ม `GRAD_ACC` เป็น 8
> **คำแนะนำ:** หากรันบน Google Colab ให้เปิด **T4 GPU** แล้วคลิก **Run all**
> หากรันบนเครื่องคอมพิวเตอร์ที่ไม่มี GPU ระบบจะตรวจจับและโหลดโมเดลที่มีอยู่ให้อัตโนมัติ (ไม่ค้างเทรนบน CPU)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, set_seed
import os
set_seed(SEED)

# ตรวจสอบว่ามีโมเดลที่เคยเทรน/บันทึกไว้แล้วหรือไม่
already_trained = os.path.exists(f"{OUTPUT_DIR}/config.json") or os.path.exists(f"../{OUTPUT_DIR}/config.json")

if already_trained and device == "cpu":
    print(f"⚡ ตรวจพบโมเดลที่เทรนเสร็จแล้วใน '{OUTPUT_DIR}' (รันบน CPU ข้ามขั้นตอนการเทรนเพื่อประหยัดเวลา)")
    trainer = None
else:
    args = Seq2SeqTrainingArguments(
        output_dir="checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC, weight_decay=0.01,
        optim="adafactor", gradient_checkpointing=True,   # ประหยัดหน่วยความจำ GPU
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        fp16=False, logging_steps=50, report_to="none", seed=SEED)

    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                             data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
    trainer.train()

⚡ ตรวจพบโมเดลที่เทรนเสร็จแล้วใน 'mt5-thaisum' (รันบน CPU ข้ามขั้นตอนการเทรนเพื่อประหยัดเวลา)


## 7. วัดผลบน test (ส่งตารางนี้ให้คนที่ 7)

In [ ]:
import os, json
import pandas as pd
from transformers import AutoModelForSeq2SeqLM

# 1. กำหนดโมเดลสำหรับทดสอบ (ใช้ trainer.model หากเพิ่งเทรนเสร็จ หรือโหลดจาก OUTPUT_DIR)
if 'trainer' in globals() and trainer is not None and hasattr(trainer, 'model'):
    eval_model = trainer.model
else:
    model_dir = OUTPUT_DIR if os.path.exists(f"{OUTPUT_DIR}/config.json") else (f"../{OUTPUT_DIR}" if os.path.exists(f"../{OUTPUT_DIR}/config.json") else MODEL_NAME)
    print(f"📦 โหลดโมเดลสำหรับทดสอบจาก: {model_dir}")
    eval_model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device)

# 2. ป้องกัน NameError หากรันข้ามเซลล์
if 'summarize' not in globals():
    def summarize(texts, model, bs=8):
        out = []
        model.eval()
        gen_kwargs = {"forced_bos_token_id": tokenizer.lang_code_to_id[MBART_LANG]} if MBART_LANG else {}
        for i in range(0, len(texts), bs):
            enc = tokenizer(texts[i:i+bs], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
            with torch.no_grad():
                ids = model.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=NUM_BEAMS,
                                     no_repeat_ngram_size=3, length_penalty=1.0, **gen_kwargs)
            out += tokenizer.batch_decode(ids, skip_special_tokens=True)
        return out

# 3. โหลดผลลัพธ์การทำนาย (หากมี predictions_mt5.csv อยู่แล้ว โหลดทันทีไม่ต้องรอคำนวณซ้ำ)
pred_file = "predictions_mt5.csv" if os.path.exists("predictions_mt5.csv") else ("../predictions_mt5.csv" if os.path.exists("../predictions_mt5.csv") else None)
if pred_file:
    print(f"✓ โหลดผลสรุปที่มีอยู่แล้วจาก {pred_file}")
    _df = pd.read_csv(pred_file)
    preds = _df["pred"].tolist()
    refs = _df["summary"].tolist()
    if 'test_df' not in globals():
        test_df = _df
else:
    preds = summarize(test_df.body.tolist(), eval_model)
    refs = test_df.summary.tolist()

# 4. โหลดหรือคำนวณคะแนน ROUGE
res_file = f"{OUTPUT_DIR}/result.json" if os.path.exists(f"{OUTPUT_DIR}/result.json") else (f"../{OUTPUT_DIR}/result.json" if os.path.exists(f"../{OUTPUT_DIR}/result.json") else None)
if res_file:
    with open(res_file, encoding="utf-8") as f:
        result = json.load(f)
else:
    scores = thai_rouge(preds, refs)
    zero_scores = thai_rouge(zero, refs[:N0]) if ('zero' in globals() and 'N0' in globals()) else {}
    train_count = len(train_df) if 'train_df' in globals() else TRAIN_SIZE
    test_count = len(test_df) if 'test_df' in globals() else len(refs)
    result = {"model": MODEL_NAME, **scores, "zero_shot_rouge_first100": zero_scores,
              "train_size": train_count, "test_size": test_count, "epochs": EPOCHS, "lr": LR,
              "max_input": MAX_INPUT, "num_beams": NUM_BEAMS}
result

📦 โหลดโมเดลสำหรับทดสอบจาก: mt5-thaisum
✓ โหลดผลสรุปที่มีอยู่แล้วจาก predictions_mt5.csv


{
  'model': 'csebuetnlp/mT5_multilingual_XLSum',
  'rouge1': 64.54,
  'rouge2': 58.5,
  'rougeL': 63.19,
  'zero_shot_rouge_first100': {'rouge1': 36.84, 'rouge2': 25.17, 'rougeL': 32.68},
  'train_size': 5000,
  'test_size': 500,
  'epochs': 2,
  'lr': 0.0001,
  'max_input': 512,
  'num_beams': 4
}

### บทสรุปทั้งหมด (ส่งให้คนที่ 6 เช็กชื่อเฉพาะ และคนที่ 7 วิเคราะห์ข้อผิดพลาด)

In [ ]:
import os
import pandas as pd
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

if 'scorer' not in globals():
    class ThaiTokenizer:
        def tokenize(self, text):
            return [w for w in word_tokenize(text, engine="newmm") if w.strip()]
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

if 'test_df' not in globals():
    pred_path = "predictions_mt5.csv" if os.path.exists("predictions_mt5.csv") else ("../predictions_mt5.csv" if os.path.exists("../predictions_mt5.csv") else None)
    if pred_path:
        test_df = pd.read_csv(pred_path)
    else:
        test_df = load(TEST_CSV, "test", TEST_SIZE)

out = test_df.assign(pred=preds)
if "rougeL" not in out.columns or out["rougeL"].isnull().any():
    out["rougeL"] = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
out.to_csv("predictions_mt5.csv", index=False)
pd.set_option("display.max_colwidth", 200)
print("บทสรุปที่ ROUGE-L ต่ำสุด 5 ข่าว:")
out.sort_values("rougeL")[["summary", "pred", "rougeL"]].head()

บทสรุปที่ ROUGE-L ต่ำสุด 5 ข่าว:


,summary,pred,rougeL
321,เห็น แอฟ-ทักษอร ทำหน้าที่ คุณแม่ แข็งขัน พา น้องปีใหม่ ไปไหนมาไหนด้วยตลอด ล่าสุดเจ้าตัวเผยถึงกระแสข่าวหย่ากับสามี สงกรานต์ เตชะณรงค์ ว่าไม่ขอพูดถึงเรื่องนี้,แอฟ-ทักษอร เตชะณรงค์ เผยถึงเรื่องราวความรักกับอดีตสามี สงกรานต์ เตชะณรงค์ พร้อมเปิดใจถึงเรื่องหัวใจว่าไม่ได้เจอกันมานานแล้ว,0.000000
133,ศ.ดร.สุรเกียรติ์ เสถียรไทย ผู้บริหารวุฒิศักดิ์ คลินิก กรุ๊ป จัดงานเปิดตัวแคมเปญใหม่ มั่นใจใช้ชีวิต พลิกโฉมองค์กรสู่การเติบโตอย่างยั่งยืน,"วุฒิศักดิ์ คลินิก กรุ๊ป เปิดตัวแคมเปญ ""มั่นใจใช้ชีวิต"" เพื่อเสริมสร้างความมั่นใจให้กับคนไทยทุกคน",0.000000
371,ชเวซุนซิล คนสนิทของ ปธน.ปาร์กกึนเฮ ปฏิเสธทุกข้อกล่าวหาในคดีอื้อฉาวทุจริตและการใช้อิทธิพลแทรกแซงการเมือง,นางชเว ซุน-ซิล คนสนิทของประธานาธิบดีปาร์ก กึน-เฮ ของเกาหลีใต้ ปฏิเสธทุกข้อกล่าวหาในศาลเกาหลีใต้,0.090909


## 8. ลองสรุปข่าวใหม่

In [ ]:
# กำหนดโมเดลที่ใช้งานสรุป
active_model = eval_model if 'eval_model' in globals() else (trainer.model if 'trainer' in globals() and trainer is not None else AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device))

news = """กรมอุตุนิยมวิทยาประกาศเตือนพายุฤดูร้อนบริเวณภาคเหนือและภาคตะวันออกเฉียงเหนือ ระหว่างวันที่ 10-12 นี้
จะมีพายุฝนฟ้าคะนอง ลมกระโชกแรง และลูกเห็บตกบางพื้นที่ ขอให้ประชาชนหลีกเลี่ยงการอยู่ในที่โล่งแจ้ง
และใต้ต้นไม้ใหญ่ ส่วนเกษตรกรควรเตรียมป้องกันความเสียหายของผลผลิต"""
summarize([news], active_model)[0]

'กรมอุตุนิยมวิทยาประกาศเตือนพายุฤดูร้อนบริเวณภาคเหนือและภาคตะวันออกเฉียงเหนือ ช่วงวันที่ 10-12 นี้'

## 9. บันทึกโมเดล
zip ไว้ดาวน์โหลด หรือคัดลอกเข้า Google Drive ของกลุ่ม · **ไม่ต้อง commit โมเดลเข้า GitHub** (ใหญ่ ~2 GB)

In [ ]:
import json, shutil, os

# บันทึกโมเดล (รองรับทั้งกรณีเทรนผ่าน trainer หรือโหลดโมเดล)
if 'trainer' in globals() and trainer is not None and hasattr(trainer, 'save_model'):
    trainer.save_model(OUTPUT_DIR)
elif 'eval_model' in globals():
    eval_model.save_pretrained(OUTPUT_DIR)
elif 'active_model' in globals():
    active_model.save_pretrained(OUTPUT_DIR)

if 'tokenizer' in globals():
    tokenizer.save_pretrained(OUTPUT_DIR)

if 'result' in globals():
    json.dump(result, open(f"{OUTPUT_DIR}/result.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("บันทึกแล้ว:", OUTPUT_DIR + ".zip")

# คัดลอกเข้า Drive (เอาคอมเมนต์ออกถ้าต้องการ)
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy(OUTPUT_DIR + ".zip", "/content/drive/MyDrive/")

บันทึกแล้ว: mt5-thaisum.zip
